# 8.1 Clustering Jerárquico Aglomerativo (`MineriaProtect` · JobPath · Sesión 9 · Guía 01)

> **Curso / Unidad:** Minería de Datos · Unidad 2 (CRISP-DM Fase 4 y Fase 5 · Descubrimiento de Estructura No Supervisada)  
> **Guía técnica implementada:** `01_agglomerative_hierarchical_clustering.md`  
> **Pregunta de negocio (Acta `1.0_comprension_negocio.md`):** *¿Qué arquetipos naturales de trayectoria laboral y de exigencia de competencias ESCO existen en el mercado laboral cuando agrupamos sin usar etiqueta supervisada, y cómo orientan las intervenciones del Observatorio Laboral?*

---

## 1. Contexto, Política Cero-Fuga (`Zero Target Leakage`) y Estrategia de Muestreo en 3 Niveles ($O(n^2)$)

A diferencia de los cuadernos supervisados (`6.0`, `7.0`, `7.1`, `7.2`), donde el objetivo era predecir `reincide = (veces_ese_oficio > 1)`, en **Clustering Jerárquico Aglomerativo** **ninguna variable objetivo ni equivalente (`reincide`, `veces_ese_oficio`, `n_returns`, `n_occ`, `exploration_ratio`) entra en la matriz `X`**. El algoritmo construye una taxonomía ascendente (*bottom-up*) fusionando paso a paso los grupos más cercanos según la distancia euclidiana sobre variables previamente estandarizadas (`StandardScaler`).

### Restricción de memoria $O(n^2)$ y estrategia de muestreo estratificado en 3 niveles
Dado que `scipy.cluster.hierarchy.linkage` requiere almacenar $\binom{n}{2} = \frac{n(n-1)}{2}$ distancias pares en coma flotante de 64 bits (`8 bytes` por par):
- Sobre las `1,738,738` experiencias clasificadas requeriría **`12,092.9 GB` (`~12.1 TB`) de RAM**.
- Sobre las `352,093` personas clasificadas requeriría **`495.9 GB` de RAM**.

Para garantizar rigor estadístico sin comprometer la memoria, implementamos el protocolo de **3 niveles** con semilla fija (`random_state = 42`):
1. **Nivel 1 · Micro-Muestra Visual ($n = 60$ personas de Train)**: estratificada por gran grupo ISCO y bandera de educación MNAR, para graficar el dendrograma completo con hojas legibles (convención de la Sesión 9).
2. **Nivel 2 · Muestra Cuantitativa de Descubrimiento ($n = 3,000$ personas y $n = 3,000$ experiencias de Train)**: ejecuta el torneo de métodos de linkage (**priorizando `ward`** frente a `complete`, `average` y `single`), evalúa los cortes horizontales $k \in \{2,\dots,8\}$ (`Silhouette Score`, `Davies-Bouldin Index`, `Calinski-Harabasz Index` y salto vertical de fusión) y grafica el dendrograma truncado (`lastp = 30`).
3. **Nivel 3 · Estabilidad en 5 Semillas (`[42, 100, 2026, 7, 999]`), Sensibilidad `Versión B` vs. `Versión A` y Proyección Completa (`NearestCentroid`)**: verifica la estabilidad muestral (`media ± DE`) y proyecta los centroides de Ward en espacio estandarizado sobre la totalidad de las **`352,093` personas** y **`1,738,738` experiencias clasificadas**.

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import IPython.display as ipd
from IPython.display import Image, Markdown, display
import numpy as np
import pandas as pd

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if candidate.name == "MineriaProtect":
        PROJECT_ROOT = candidate
        break
    if (candidate / "MineriaProtect").is_dir():
        PROJECT_ROOT = candidate / "MineriaProtect"
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.ScriptsCsv.ClusteringJerarquico import (
    HierarchicalClusteringConfig,
    VERSION_A,
    VERSION_B,
    V_EXPERIENCE,
    V_PERSON,
    append_bitacora_jerarquico_md,
    append_comparacion_ab_jerarquico_md,
    assert_zero_leakage_in_feature_lists,
    build_data_usage_and_sampling_table,
    build_hardware_feasibility_table,
    execute_hierarchical_view,
    export_all_hierarchical_tables,
    export_ram_expansion_tables,
    generate_all_hierarchical_figures,
    load_and_prepare_clustering_dataset,
    plot_ward_expansion_stability_es,
    run_five_seed_stability_and_version_ab_sensitivity,
    run_ram_optimized_ward_expansion,
    snapshot_input_hashes,
    verify_input_hashes,
    verify_narrative_numbers_against_tables,
    write_comparative_analysis_8_0_vs_8_1_md,
)

CONFIG = HierarchicalClusteringConfig.default()
INITIAL_HASHES = snapshot_input_hashes(CONFIG)
CURRENT_RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
CURRENT_RUN_ID = "RUN-" + datetime.now().strftime("%Y%m%d-%H%M%S")

leakage_audit_df = assert_zero_leakage_in_feature_lists()
hardware_feasibility_df = build_hardware_feasibility_table()
print(f"Run ID: {CURRENT_RUN_ID} | Timestamp: {CURRENT_RUN_TIMESTAMP}")
display(Markdown("### 1.1 Auditoría Cero-Fuga (`Zero Target Leakage`) y Anti-Colinealidad (`E = C + K`)"))
display(leakage_audit_df)
display(Markdown("### 1.2 Justificación Formal de Factibilidad de Hardware ($O(n^2)$ en Acer Nitro ANV15-52 · 32 GB RAM)"))
display(hardware_feasibility_df)

Run ID: RUN-20261120-073736 | Timestamp: 2026-11-20 07:37:36


### 1.1 Auditoría Cero-Fuga (`Zero Target Leakage`) y Anti-Colinealidad (`E = C + K`)

vista_analitica,n_variables_clustering,variables_incluidas,columnas_fuga_detectadas,colinealidad_E_igual_C_mas_K_evitada,tratamiento_educacion_mnar,estado_auditoria
Vista Primaria · Personas / Trayectorias (V_PERSON),9,"n_jobs, span_q, dur_mean, overlap_ratio, mean_skills_competence, mean_skills_knowledge, mean_prop_competence, nivel_academico_ord_imputado, educacion_no_reportada",0,"Sí (usa competence, knowledge y prop_competence)",Bandera educacion_no_reportada (0/1) + ordinal imputado con mediana de Train (2.0); sin relleno 0,APROBADO (100% no supervisado)
Vista Secundaria · Experiencias / Perfiles ESCO (V_EXPERIENCE),7,"dur_q, job_order, n_skills_competence, n_skills_knowledge, prop_competence_essential, nivel_academico_ord_imputado, educacion_no_reportada",0,"Sí (usa competence, knowledge y prop_competence_essential)",Bandera educacion_no_reportada (0/1) + ordinal imputado con mediana de Train (2.0); sin relleno 0,APROBADO (100% no supervisado)


### 1.2 Justificación Formal de Factibilidad de Hardware ($O(n^2)$ en Acer Nitro ANV15-52 · 32 GB RAM)

vista_o_escenario,filas_n,pares_condensados_n_n1_div_2,ram_matriz_condensada_gb_float64,ram_matriz_condensada_gib_binario,pico_ram_estimado_windows_scipy_gb,factible_en_32gb_ram
V_EXPERIENCE Train (100%),1391018,967464842653,7739.719,7208.175,15479.44,No (MemoryError inmediato; supera RAM física en >240x)
V_PERSON Total Clasificado (100%),352093,61984564278,495.877,461.821,991.75,No (MemoryError inmediato; supera RAM física en >15x)
V_PERSON Train (100%),281674,39669980301,317.360,295.564,634.72,No (MemoryError inmediato; supera RAM física en 10x)
V_PERSON Muestra Anterior (8.1 inicial),3000,4498500,0.036,0.034,1.20,Sí (Muy holgado; < 1 s en SciPy)
V_PERSON Muestra Optimizada (Seleccionada),25000,312487500,2.500,2.328,8.00,Sí (Óptima y segura en 32 GB RAM; deja >20 GB libres)
V_PERSON Límite Superior Seguro,30000,449985000,3.600,3.353,11.52,Sí (Viable con margen ajustado si hay otras apps abiertas)


---

## 2. Carga Defensiva de `Versión B` (`sin_outliers`, Primaria) y `Versión A` (`con_outliers`, Sensibilidad) y Tabla de Uso de Datos

Cargamos ambos datasets preprocesados desde `data/03_processed/`, filtramos en RAM las `1,738,738` experiencias con ocupación ESCO clasificada (`352,093` personas únicas), aplicamos el mapa oficial de partición agrupada por persona (`reports/eda_jobpath/tables/split_map_resume_id.csv`) y construimos la tabla maestra de **Uso de datos y estrategia de muestreo**.

In [2]:
dataset_b = load_and_prepare_clustering_dataset(
    csv_path=CONFIG.clean_version_b_path,
    split_map_path=CONFIG.split_map_path,
    version_code=VERSION_B,
)
dataset_a = load_and_prepare_clustering_dataset(
    csv_path=CONFIG.clean_version_a_path,
    split_map_path=CONFIG.split_map_path,
    version_code=VERSION_A,
)

data_usage_df = build_data_usage_and_sampling_table(
    dataset_b=dataset_b,
    config=CONFIG,
    exp_sample_rows_3000=CONFIG.discovery_sample_n,
)

display(Markdown("### 2.1 Tabla Maestra de Uso de Datos y Estrategia de Muestreo Jerárquico en 3 Niveles"))
display(data_usage_df)

### 2.1 Tabla Maestra de Uso de Datos y Estrategia de Muestreo Jerárquico en 3 Niveles

etapa_uso_de_datos,filas_experiencias,personas_unicas,pct_sobre_total_csv,pct_sobre_clasificados,memoria_matriz_distancias_condensada,detalle_y_justificacion
1. Universo bruto en CSV de entrada (Versión B y Versión A),1871014,355308,100.00%,no aplica,"14,002.6 GB (inviable en RAM para linkage O(n^2))",Total de experiencias laborales antes de filtrar ocupaciones sin código ESCO estandarizado.
2. Filas excluidas en RAM (es_ocupacion_clasificada == 0),132276,3215,7.07%,no aplica,no aplica,Excluidas en RAM antes de clusterizar (Decisión D07 de Fase 3) por carecer de perfil ESCO (128981 con emparejado='unknown' y 3295 con 'rescatado' sin padre ESCO; 3215 personas exclusivas).
3. Universo clasificado modelado (es_ocupacion_clasificada == 1),1738738,352093,92.93%,100.00%,"12,092.9 GB (experiencias) / 495.9 GB (personas)","Universo válido alineado 1 a 1 entre Versión B (sin outliers IQR, primaria) y Versión A (con outliers, sensibilidad)."
4. Partición Train oficial agrupada por persona (80% · resume_id),1391018,281674,74.35%,80.00%,"7,739.5 GB (experiencias) / 317.4 GB (personas)","Partición de entrenamiento (seed=42, 0 personas compartidas con Test) donde se ajusta StandardScaler y se extraen las muestras jerárquicas."
5. Partición Test oficial agrupada por persona (20% · resume_id),347720,70419,18.58%,20.00%,483.7 GB (experiencias) / 19.8 GB (personas),Partición retenida (20%) para validar estabilidad out-of-sample mediante proyección NearestCentroid en espacio estandarizado.
6. Educación no reportada (MNAR · educacion_no_reportada == 1) en Train,160664,34266,8.59%,11.55%,no aplica,"Registros de Train sin grado académico reportado (codificados con bandera binaria 1 + mediana ordinal de Train=2.0, sin rellenar jamás con 0)."
7. Educación no reportada (MNAR · educacion_no_reportada == 1) en Test,40270,8618,2.15%,11.58%,no aplica,Registros de Test sin grado académico reportado (mantenidos con bandera binaria 1 + mediana ordinal de Train=2.0).
8. Educación no reportada (MNAR · educacion_no_reportada == 1) en Total Clasificado,200934,42884,10.74%,11.56%,no aplica,"Total de registros clasificados con educación no reportada (Train + Test = 200,934 filas; 44,568 personas)."
9. Nivel 1 de Muestreo Jerárquico · Micro-Muestra Visual (n = 60 personas Train),60,60,0.02%,0.02%,"0.014 MB (1,770 pares · instantáneo)",Submuestra estratificada por gran grupo ISCO y bandera MNAR (seed=42) para graficar el dendrograma completo con hojas legibles (convención Sesión 9).
"10. Nivel 2 de Muestreo Jerárquico · Muestra de Descubrimiento (n = 3,000 personas Train)",3000,3000,0.16%,0.17%,"36.0 MB (4,498,500 pares · ~0.8 s en SciPy)","Muestra cuantitativa de descubrimiento (3,000 personas Train y submuestra de 3,000 experiencias Train, seed=42) para torneo de linkage y corte óptimo k*."


---

## 3. Torneo de Métodos de Linkage (`ward` vs. `complete`, `average`, `single`), Dendrogramas y Corte Óptimo ($k^*$)

Ejecutamos el motor jerárquico sobre las dos vistas analíticas en **`Versión B · SIN outliers (Primaria)`**:
1. **Vista Primaria · Personas / Trayectorias (`V_PERSON`, `352,093` personas)**: evalúa la forma de la carrera (`n_jobs`, `span_q`, `dur_mean`, `overlap_ratio`), la exigencia promedio ESCO (`mean_skills_competence`, `mean_skills_knowledge`, `mean_prop_competence`) y el nivel educativo (`nivel_academico_ord_imputado`, `educacion_no_reportada`).
2. **Vista Secundaria · Experiencias Laborales (`V_EXPERIENCE`, `1,738,738` filas)**: evalúa cada puesto de trabajo según su duración (`dur_q`), etapa de carrera (`job_order`), perfil ESCO y educación.

In [3]:
person_bundle = execute_hierarchical_view(
    df_full=dataset_b.df_persons,
    feature_cols=V_PERSON,
    strata_cols=["isco_major_modal_posthoc", "educacion_no_reportada"],
    view_name="person",
    config=CONFIG,
    posthoc_target_col="any_repeated_posthoc",
    posthoc_isco_col="isco_major_modal_posthoc",
)

exp_bundle = execute_hierarchical_view(
    df_full=dataset_b.df_experiences,
    feature_cols=V_EXPERIENCE,
    strata_cols=["isco_major_1d", "educacion_no_reportada"],
    view_name="experience",
    config=CONFIG,
    posthoc_target_col="reincide",
    posthoc_isco_col="isco_major_1d",
)

display(Markdown("### 3.1 Torneo de Métodos de Linkage en Muestra Estandarizada de Personas (`n = 3,000` · `k = 4`)"))
display(person_bundle.linkage_tournament_df)

display(Markdown("### 3.2 Evaluación de Cortes Horizontales del Dendrograma Ward (`k = 2..8` · Saltos Verticales, Silueta y Davies-Bouldin)"))
display(person_bundle.cuts_evaluation_df)

print(
    f"Concordancia en muestra de descubrimiento (n={CONFIG.discovery_sample_n}) entre Ward (k={person_bundle.k_opt}) y K-Means (k={person_bundle.k_opt}): "
    f"ARI = {person_bundle.ari_vs_kmeans:.4f} | NMI = {person_bundle.nmi_vs_kmeans:.4f}"
)

Concordancia en muestra de descubrimiento (n=3000) entre Ward (k=4) y K-Means (k=4): ARI = 0.4164 | NMI = 0.5014


### 3.1 Torneo de Métodos de Linkage en Muestra Estandarizada de Personas (`n = 3,000` · `k = 4`)

metodo_linkage,correlacion_cofenetica,k_evaluado,silhouette_score,davies_bouldin_index,calinski_harabasz_index,tamano_min_cluster,tamano_max_cluster,pct_cluster_mas_grande,decision_y_diagnostico_es
ward,0.4463,4,0.1827,1.9489,370.72,160,1650,55.00,SELECCIONADO (prioridad metodológica: minimiza incremento de varianza intra-clúster ΔESS y produce segmentos balanceados y accionables)
complete,0.5623,4,0.2249,1.3672,210.72,38,2701,90.03,Descartado por efecto encadenamiento (chaining: el clúster mayor absorbe 90.03% de la muestra y aísla hojas marginales)
average,0.7348,4,0.4431,0.8927,121.36,2,2925,97.50,Descartado por efecto encadenamiento (chaining: el clúster mayor absorbe 97.50% de la muestra y aísla hojas marginales)
single,0.5010,4,0.5048,0.4334,10.53,1,2996,99.87,Descartado por efecto encadenamiento (chaining: el clúster mayor absorbe 99.87% de la muestra y aísla hojas marginales)


### 3.2 Evaluación de Cortes Horizontales del Dendrograma Ward (`k = 2..8` · Saltos Verticales, Silueta y Davies-Bouldin)

k_clusters,altura_fusion_superior_ward,altura_fusion_inferior_ward,salto_vertical_fusion,altura_corte_horizontal,silhouette_score,davies_bouldin_index,calinski_harabasz_index,tamano_min_cluster,pct_min_cluster,tamano_max_cluster,pct_max_cluster,justificacion_corte_es
2,76.439,71.141,5.297,73.790,0.2204,1.5182,363.74,341,11.37,2659,88.63,Partición binaria macro demasiado gruesa (separa solo 2 grandes bloques sin distinguir rotación vs. complejidad ESCO)
3,71.141,60.951,10.190,66.046,0.1803,2.2490,379.14,341,11.37,1650,55.00,"Partición parsimoniosa válida, pero fusiona a los especialistas estables con el grupo de alta exigencia ESCO"
4,60.951,50.891,10.060,55.921,0.1827,1.9489,370.72,160,5.33,1650,55.00,"CORTE ÓPTIMO SELECCIONADO (k=4: equilibrio entre salto vertical de fusión=10.06, Silueta=0.1827, Davies-Bouldin=1.9489 y 4 arquetipos accionables con soporte mínimo=5.33%)"
5,50.891,49.460,1.431,50.176,0.1812,1.8463,350.22,160,5.33,1650,55.00,Descartado por sobre-fragmentación operativa (k=5 divide arquetipos en subgrupos con menor salto vertical=1.43)
6,49.460,48.624,0.837,49.042,0.1462,1.8035,342.69,160,5.33,1338,44.60,Descartado por sobre-fragmentación operativa (k=6 divide arquetipos en subgrupos con menor salto vertical=0.84)
7,48.624,44.783,3.840,46.703,0.1521,1.6632,343.46,140,4.67,1338,44.60,Descartado por sobre-fragmentación operativa (k=7 divide arquetipos en subgrupos con menor salto vertical=3.84)
8,44.783,36.872,7.912,40.827,0.1200,1.6812,342.58,140,4.67,810,27.00,Descartado por sobre-fragmentación operativa (k=8 divide arquetipos en subgrupos con menor salto vertical=7.91)


---

## 4. Estabilidad en 5 Semillas Estratificadas (`42, 100, 2026, 7, 999`) y Sensibilidad `Versión B` (`sin_outliers`) vs. `Versión A` (`con_outliers`)

Evaluamos si la estructura jerárquica de Ward y sus métricas internas (`Correlación Cofenética`, `Silhouette Score`, `Davies-Bouldin Index`, `Calinski-Harabasz Index`) se mantienen estables a través de 5 muestras estratificadas independientes de `3,000` personas de Train, y medimos el impacto de los outliers comparando exactamente las mismas personas en **`Versión B · SIN outliers`** frente a **`Versión A · CON outliers`**.

In [4]:
per_seed_df, ab_summary_df = run_five_seed_stability_and_version_ab_sensitivity(
    dataset_b=dataset_b,
    dataset_a=dataset_a,
    config=CONFIG,
)

display(Markdown("### 4.1 Detalle Semilla por Semilla (`42, 100, 2026, 7, 999` · `n = 3,000` Personas por Semilla)"))
display(per_seed_df)

display(Markdown("### 4.2 Resumen Comparativo `Versión B (SIN outliers)` vs. `Versión A (CON outliers)` (Media ± DE)"))
display(ab_summary_df)

### 4.1 Detalle Semilla por Semilla (`42, 100, 2026, 7, 999` · `n = 3,000` Personas por Semilla)

semilla,n_personas_muestra,k_optimo,cophenet_version_b,cophenet_version_a,silhouette_version_b,silhouette_version_a,delta_silhouette_b_menos_a,davies_bouldin_version_b,davies_bouldin_version_a,calinski_harabasz_version_b,calinski_harabasz_version_a,altura_corte_ward_b,altura_corte_ward_a,pct_max_cluster_b,pct_max_cluster_a,ari_concordancia_b_vs_a,nmi_concordancia_b_vs_a
42,3000,4,0.4463,0.3828,0.1827,0.1619,0.0208,1.9489,1.8595,370.72,360.18,55.921,56.111,55.00,45.13,0.2140,0.3602
100,3000,4,0.3936,0.3976,0.1263,0.1275,-0.0012,2.0033,1.9094,379.15,382.88,59.345,60.568,34.37,34.53,0.4593,0.5182
2026,3000,4,0.4267,0.4506,0.1460,0.1812,-0.0352,2.0110,1.8314,412.24,405.72,61.219,63.721,35.43,50.60,0.3722,0.4777
7,3000,4,0.4821,0.4218,0.1970,0.1660,0.0310,1.6890,1.8950,387.88,391.85,60.480,59.243,65.23,46.50,0.3464,0.4482
999,3000,4,0.3823,0.3690,0.1277,0.1472,-0.0194,2.0426,1.8751,353.89,381.06,57.772,59.785,42.50,37.53,0.4112,0.4780


### 4.2 Resumen Comparativo `Versión B (SIN outliers)` vs. `Versión A (CON outliers)` (Media ± DE)

version_datos,n_semillas,personas_por_semilla,k_optimo,cophenet_media,cophenet_std,silhouette_media,silhouette_std,davies_bouldin_media,davies_bouldin_std,calinski_harabasz_media,altura_corte_ward_media,pct_max_cluster_media,ari_b_vs_a_media,ari_b_vs_a_std,nmi_b_vs_a_media,decision_version_es
Versión B · SIN outliers (Primaria),5,3000,4,0.4262,0.0361,0.1559,0.0289,1.9390,0.1286,380.78,58.947,46.51,0.3606,0.0826,0.4565,RECOMENDADA COMO PRIMARIA (acota colas extremas de conteos ESCO hasta 47.5/39.5/13.5 evitando que outliers arrastren la varianza de Ward)
Versión A · CON outliers (Sensibilidad),5,3000,4,0.4044,0.0290,0.1568,0.0182,1.8741,0.0273,384.34,59.886,42.86,0.3606,0.0826,0.4565,Mantenida como contraste de sensibilidad (las colas sin recortar hasta 129 competencias inflan las distancias euclidianas de las hojas extremas)


---

### 4.3 Muestreo Estratificado Optimizado para 32 GB RAM (`n = 25,000` Personas) y Auditoría de Estabilidad frente a `n = 3,000` y K-Means (`8.0`)

Siguiendo la tabla de factibilidad $O(n^2)$ de la Sección 1.2 (Acer Nitro ANV15-52 · Intel Core i5-13420H · **32 GB RAM**, `31.7 GB` útiles), el tamaño seguro óptimo para maximizar la cobertura de `V_PERSON` Train sin arriesgar un `MemoryError` en Windows 11 es **`n = 25,000` personas** (`8.88%` de todo Train, `8.33x` mayor que la muestra inicial de `3,000`):
- **Matriz condensada `float64`:** $\frac{25000 \times 24999}{2} \times 8\text{ bytes} = \mathbf{2.500\text{ GB}}$ (`2.328 GiB`), con un pico total seguro de `~8–10 GB` de RAM.
- **Estratificación reproducible (`random_state = 42`):** estratificada por Gran Grupo ISCO (`isco_major_modal_posthoc`) y bandera MNAR (`educacion_no_reportada`), ajustando `StandardScaler` + `scipy.cluster.hierarchy.linkage(method="ward")` + `fcluster(t=4)` y proyectando a las **`352,093` personas** mediante `NearestCentroid`.

In [5]:
expansion_result = run_ram_optimized_ward_expansion(
    df_persons=dataset_b.df_persons,
    person_bundle_3k=person_bundle,
    config=CONFIG,
    n_expanded=25000,
)

expansion_fig_path = plot_ward_expansion_stability_es(
    expansion=expansion_result,
    person_bundle_3k=person_bundle,
    config=CONFIG,
)
expansion_tables = export_ram_expansion_tables(
    expansion=expansion_result,
    config=CONFIG,
)

print(
    f"Ajuste Ward n=25,000 completado en {expansion_result.fit_seconds:.2f} s | "
    f"Estabilidad n=25,000 vs. n=3,000 (en 352,093 personas): ARI = {expansion_result.ari_25k_vs_3k_full:.4f}, NMI = {expansion_result.nmi_25k_vs_3k_full:.4f} | "
    f"Concordancia n=25,000 vs. K-Means 8.0: ARI = {expansion_result.ari_25k_vs_kmeans_full:.4f}, NMI = {expansion_result.nmi_25k_vs_kmeans_full:.4f}"
)

display(Markdown("#### Tabla 4.3.1 · Estabilidad y Métricas: Ward `n = 3,000` vs. Ward Optimizado `n = 25,000` vs. K-Means `8.0` (`100% Train`)"))
display(expansion_result.stability_comparison_df)

display(Markdown("#### Tabla 4.3.2 · Perfiles en Escala Original del Modelo Ward Optimizado (`n = 25,000` → `352,093` Personas)"))
display(expansion_result.profile_25k_df)

display(Markdown(f"#### Figura: `{expansion_fig_path.name}`"))
display(Image(filename=str(expansion_fig_path)))

Ajuste Ward n=25,000 completado en 146.11 s | Estabilidad n=25,000 vs. n=3,000 (en 352,093 personas): ARI = 0.3582, NMI = 0.5035 | Concordancia n=25,000 vs. K-Means 8.0: ARI = 0.6502, NMI = 0.7296


#### Tabla 4.3.1 · Estabilidad y Métricas: Ward `n = 3,000` vs. Ward Optimizado `n = 25,000` vs. K-Means `8.0` (`100% Train`)

modelo_y_muestra,n_entrenamiento,pct_train_usado,silhouette_score,davies_bouldin_index,calinski_harabasz_index,pct_cluster_0,pct_cluster_1,pct_cluster_2,pct_cluster_3,ari_vs_ward_n25000_en_352093,nmi_vs_ward_n25000_en_352093,ari_vs_kmeans_8_0_en_352093,nmi_vs_kmeans_8_0_en_352093,tiempo_ajuste_seg
"8.1 · Ward Jerárquico (Muestra n = 3,000 → Proyectado a 352,093)",3000,1.07,0.1827,1.9489,370.72,52.13,11.65,9.47,26.75,0.3582,0.5035,0.4708,0.5462,0.80
"8.1 · Ward Jerárquico Optimizado (Muestra n = 25,000 → Proyectado a 352,093)",25000,8.88,0.1167,1.9295,3134.50,34.01,11.82,38.85,15.32,1.0000,1.0000,0.6502,0.7296,146.11
"8.0 · K-Means++ (100% Train n = 281,674 → Evaluado en 352,093)",281674,100.00,0.1948,1.6298,46780.83,43.90,11.66,27.99,16.44,0.6502,0.7296,1.0000,1.0000,1.80


#### Tabla 4.3.2 · Perfiles en Escala Original del Modelo Ward Optimizado (`n = 25,000` → `352,093` Personas)

cluster,nombre_segmento_es,n_muestra_descubrimiento,pct_muestra_descubrimiento,n_total_proyectado,pct_total_proyectado,n_test_proyectado,pct_test_proyectado,n_jobs_media_muestra,n_jobs_media_total,span_q_media_muestra,span_q_media_total,dur_mean_media_muestra,dur_mean_media_total,overlap_ratio_media_muestra,overlap_ratio_media_total,mean_skills_competence_media_muestra,mean_skills_competence_media_total,mean_skills_knowledge_media_muestra,mean_skills_knowledge_media_total,mean_prop_competence_media_muestra,mean_prop_competence_media_total,nivel_academico_ord_imputado_media_muestra,nivel_academico_ord_imputado_media_total,educacion_no_reportada_media_muestra,educacion_no_reportada_media_total,mean_skills_essential_media_muestra,mean_skills_essential_media_total,tasa_reincide_muestra_posthoc_pct,lift_reincide_muestra_posthoc,tasa_reincide_total_posthoc_pct,tasa_reincide_test_posthoc_pct,lift_reincide_test_posthoc,gran_grupo_isco_dominante_posthoc,accion_observatorio_recomendada_es
0,Inserción Temprana y Trayectorias Cortas en Oficios Operativos,6818,27.27,119736,34.01,23728,33.70,3.96,4.26,37.82,37.73,10.36,9.24,0.99,0.99,17.93,18.02,2.35,2.26,0.88,0.88,1.22,1.27,0.00,0.00,20.29,20.31,41.55,0.904,44.12,44.04,0.964,5 · Trabajadores de los servicios y vendedores,"Programa Puente al Primer Empleo: orientación vocacional temprana, certificación de competencias prácticas base y seguimiento en los primeros 4 trimestres post-egreso."
1,Especialistas Estables de Permanencia Prolongada,2966,11.86,41607,11.82,8356,11.87,4.71,4.70,59.88,58.02,14.70,14.13,1.04,1.03,18.03,18.13,3.34,3.31,0.84,0.84,2.00,2.00,1.00,1.00,21.42,21.46,45.92,0.999,45.59,44.96,0.984,5 · Trabajadores de los servicios y vendedores,Ruta de Actualización Continua (Up-Skilling): micro-credenciales de especialización técnica para evitar obsolescencia en carreras de larga duración.
2,Profesionales Intensivos en Conocimiento y Alta Exigencia ESCO,11924,47.70,136804,38.85,27495,39.04,5.52,5.50,46.96,44.08,10.37,9.58,1.14,1.13,18.72,18.95,4.66,5.17,0.80,0.78,2.10,2.22,0.00,0.00,23.43,24.18,48.69,1.060,47.83,47.96,1.050,2 · Profesionales científicos e intelectuales,"Vinculación I+D y Posgrados Aplicados: articulación con maestrías, liderazgo técnico y transferencia tecnológica universidad-empresa."
3,Trayectorias de Alta Rotación y Multi-Empleo Solapado,3292,13.17,53946,15.32,10840,15.39,5.09,5.20,109.66,113.00,37.96,39.35,1.64,1.63,18.30,17.74,3.98,3.88,0.82,0.82,1.38,1.43,0.02,0.02,22.31,21.66,45.14,0.982,43.96,43.99,0.963,3 · Técnicos y profesionales de nivel medio,Programa de Estabilización y Consolidación de Portafolio Laboral: acompañamiento para formalizar trayectorias fragmentadas y capitalizar la experiencia acumulada.


#### Figura: `08_estabilidad_muestra_3000_vs_25000_ward_y_kmeans.png`

[Image: C:\Users\monoc\Desktop\mineria\MineriaProtect\reports\clustering_jerarquico_jobpath\figures\08_estabilidad_muestra_3000_vs_25000_ward_y_kmeans.png]

---

## 5. Evaluación e Interpretación de Perfiles en Escala Original (`Retrato Robot`), Validación Post-Hoc con `reincide` y Galería Visual

Siguiendo las reglas de la Sesión 9:
1. **Narrar siempre en unidades reales originales (`groupby("cluster")[V].mean()`)**, nunca en puntajes $z$ estandarizados.
2. **Evaluar la variable objetivo (`reincide` / `any_repeated_posthoc`) exclusivamente *post-hoc*** para verificar si los segmentos naturales de trayectoria también estratifican la propensión a repetir oficio (`Lift` frente a la tasa base).
3. **Asignar nombres accionables en español (`Retrato Robot`)** y una intervención concreta del Observatorio Laboral a cada segmento.

In [6]:
figure_paths = generate_all_hierarchical_figures(
    person_bundle=person_bundle,
    exp_bundle=exp_bundle,
    per_seed_df=per_seed_df,
    ab_summary_df=ab_summary_df,
    config=CONFIG,
)

exported_tables = export_all_hierarchical_tables(
    data_usage_df=data_usage_df,
    person_bundle=person_bundle,
    exp_bundle=exp_bundle,
    per_seed_df=per_seed_df,
    ab_summary_df=ab_summary_df,
    config=CONFIG,
)

display(Markdown("### 5.1 Perfiles en Escala Original y Acción Recomendada · Vista Primaria Personas (`352,093` Personas Clasificadas)"))
display(person_bundle.profile_df)

display(Markdown("### 5.2 Perfiles en Escala Original y Acción Recomendada · Vista Secundaria Experiencias (`1,738,738` Experiencias Clasificadas)"))
display(exp_bundle.profile_df)

for fig_p in figure_paths:
    display(Markdown(f"#### Figura: `{fig_p.name}`"))
    display(Image(filename=str(fig_p)))

display(Markdown("### 5.3 Resumen Ejecutivo Comparativo: K-Means (`8.0`) vs. Jerárquico Ward (`8.1` · `n = 3,000` y `n = 25,000`)"))
display(expansion_result.executive_comparison_df)

### 5.1 Perfiles en Escala Original y Acción Recomendada · Vista Primaria Personas (`352,093` Personas Clasificadas)

cluster,nombre_segmento_es,n_muestra_descubrimiento,pct_muestra_descubrimiento,n_total_proyectado,pct_total_proyectado,n_test_proyectado,pct_test_proyectado,n_jobs_media_muestra,n_jobs_media_total,span_q_media_muestra,span_q_media_total,dur_mean_media_muestra,dur_mean_media_total,overlap_ratio_media_muestra,overlap_ratio_media_total,mean_skills_competence_media_muestra,mean_skills_competence_media_total,mean_skills_knowledge_media_muestra,mean_skills_knowledge_media_total,mean_prop_competence_media_muestra,mean_prop_competence_media_total,nivel_academico_ord_imputado_media_muestra,nivel_academico_ord_imputado_media_total,educacion_no_reportada_media_muestra,educacion_no_reportada_media_total,mean_skills_essential_media_muestra,mean_skills_essential_media_total,tasa_reincide_muestra_posthoc_pct,lift_reincide_muestra_posthoc,tasa_reincide_total_posthoc_pct,tasa_reincide_test_posthoc_pct,lift_reincide_test_posthoc,gran_grupo_isco_dominante_posthoc,accion_observatorio_recomendada_es
0,Inserción Temprana y Trayectorias Cortas en Oficios Operativos,1650,55.00,183538,52.13,36614,51.99,4.44,4.42,34.48,32.64,7.92,8.00,0.99,1.01,18.74,18.76,3.41,3.16,0.84,0.85,1.61,1.63,0.00,0.00,22.17,21.96,43.09,0.953,43.28,43.53,0.953,5 · Trabajadores de los servicios y vendedores,"Programa Puente al Primer Empleo: orientación vocacional temprana, certificación de competencias prácticas base y seguimiento en los primeros 4 trimestres post-egreso."
1,Especialistas Estables de Permanencia Prolongada,341,11.37,41006,11.65,8225,11.68,4.64,4.71,61.33,59.40,14.74,15.04,1.00,1.04,18.27,18.23,3.12,3.15,0.85,0.85,2.00,2.00,1.00,1.00,21.43,21.40,42.23,0.934,45.53,45.11,0.988,5 · Trabajadores de los servicios y vendedores,Ruta de Actualización Continua (Up-Skilling): micro-credenciales de especialización técnica para evitar obsolescencia en carreras de larga duración.
2,Profesionales Intensivos en Conocimiento y Alta Exigencia ESCO,160,5.33,33350,9.47,6768,9.61,3.65,3.69,51.22,45.97,17.28,14.69,1.06,1.04,14.38,16.13,8.52,7.69,0.61,0.66,2.31,2.32,0.11,0.04,23.73,23.99,33.75,0.747,34.40,34.34,0.752,2 · Profesionales científicos e intelectuales,"Vinculación I+D y Posgrados Aplicados: articulación con maestrías, liderazgo técnico y transferencia tecnológica universidad-empresa."
3,Trayectorias de Alta Rotación y Multi-Empleo Solapado,849,28.30,94199,26.75,18812,26.71,6.31,6.49,88.56,96.59,26.35,27.10,1.50,1.51,18.43,18.40,3.84,3.81,0.82,0.83,1.79,1.68,0.01,0.01,22.31,22.23,52.65,1.165,54.53,54.17,1.186,3 · Técnicos y profesionales de nivel medio,Programa de Estabilización y Consolidación de Portafolio Laboral: acompañamiento para formalizar trayectorias fragmentadas y capitalizar la experiencia acumulada.


### 5.2 Perfiles en Escala Original y Acción Recomendada · Vista Secundaria Experiencias (`1,738,738` Experiencias Clasificadas)

cluster,nombre_segmento_es,n_muestra_descubrimiento,pct_muestra_descubrimiento,n_total_proyectado,pct_total_proyectado,n_test_proyectado,pct_test_proyectado,dur_q_media_muestra,dur_q_media_total,job_order_media_muestra,job_order_media_total,n_skills_competence_media_muestra,n_skills_competence_media_total,n_skills_knowledge_media_muestra,n_skills_knowledge_media_total,prop_competence_essential_media_muestra,prop_competence_essential_media_total,nivel_academico_ord_imputado_media_muestra,nivel_academico_ord_imputado_media_total,educacion_no_reportada_media_muestra,educacion_no_reportada_media_total,n_skills_essential_media_muestra,n_skills_essential_media_total,tasa_reincide_muestra_posthoc_pct,lift_reincide_muestra_posthoc,tasa_reincide_total_posthoc_pct,tasa_reincide_test_posthoc_pct,lift_reincide_test_posthoc,gran_grupo_isco_dominante_posthoc,accion_observatorio_recomendada_es
0,Experiencias Iniciales de Corta Duración y Exigencia Básica,1790,59.67,915690,52.66,182949,52.61,8.29,7.93,3.83,3.78,17.50,17.47,2.40,1.92,0.87,0.89,1.64,1.58,0.00,0.00,19.93,19.41,32.57,1.061,31.39,31.32,1.045,5 · Trabajadores de los servicios y vendedores,Módulo de Nivelación de Competencias Transversales para reducir la rotación temprana.
1,Puestos Operativos y Administrativos con Educación No Reportada (MNAR),337,11.23,192984,11.10,38636,11.11,12.25,11.50,3.60,3.68,18.87,18.20,3.19,3.24,0.85,0.84,2.00,2.00,1.00,1.00,22.10,21.45,29.97,0.976,31.11,30.87,1.030,5 · Trabajadores de los servicios y vendedores,Certificación por Evaluación de Saberes Previos (reconocimiento de experiencia empírica sin título reportado).
2,Ocupaciones Técnicas y Profesionales de Alta Complejidad ESCO,694,23.13,507358,29.18,101676,29.24,9.72,9.00,3.78,3.98,20.21,20.46,7.34,7.39,0.71,0.71,1.96,2.03,0.00,0.00,27.57,27.92,28.39,0.925,28.65,28.67,0.957,2 · Profesionales científicos e intelectuales,Alineación Curricular con Estándares ESCO de Alta Complejidad en programas de pregrado y posgrado.
3,Empleos de Larga Permanencia o Etapa Avanzada de Carrera,179,5.97,122706,7.06,24459,7.03,67.84,67.51,3.59,3.19,20.52,18.53,3.92,3.54,0.85,0.83,1.74,1.64,0.05,0.05,24.39,22.11,22.35,0.728,23.50,23.89,0.797,3 · Técnicos y profesionales de nivel medio,Red de Mentores Egresados y Reconversión Ejecutiva para trayectorias sénior.


#### Figura: `01_dendrograma_micro_muestra_n60_ward.png`

[Image: C:\Users\monoc\Desktop\mineria\MineriaProtect\reports\clustering_jerarquico_jobpath\figures\01_dendrograma_micro_muestra_n60_ward.png]

... (10 outputs left out)

#### Figura: `07_sensibilidad_version_b_vs_version_a_5_semillas.png`

[Image: C:\Users\monoc\Desktop\mineria\MineriaProtect\reports\clustering_jerarquico_jobpath\figures\07_sensibilidad_version_b_vs_version_a_5_semillas.png]

### 5.3 Resumen Ejecutivo Comparativo: K-Means (`8.0`) vs. Jerárquico Ward (`8.1` · `n = 3,000` y `n = 25,000`)

cluster,perfil_comportamental,kmeans_8_0_tamano_y_reincide,ward_8_1_n3000_tamano_y_reincide,ward_8_1_n25000_tamano_y_reincide,isco_dominante_km_vs_ward25k,patron_reaccion_y_explicacion_es
Clúster 0,Inserción Temprana y Trayectorias Cortas en Oficios Operativos,43.90% · 47.78% (1.046x),52.13% · 43.53% (0.953x),34.01% · 44.04% (0.964x),5 · Trabajadores de los servicios y vendedores / 5 · Trabajadores de los servicios y vendedores,"Núcleo mayoritario de carreras cortas en servicios y ventas (ISCO 5). En K-Means concentra la mayor tasa post-hoc de reincidencia horizontal (cambio de empleador dentro del mismo código ESCO), mientras que Ward absorbe parte de los operativos con menor solapamiento."
Clúster 1,Trabajadores con Educación No Reportada (MNAR) y Perfil Operativo Estable,11.66% · 45.00% (0.985x),11.65% · 45.11% (0.988x),11.82% · 44.96% (0.984x),5 · Trabajadores de los servicios y vendedores / 5 · Trabajadores de los servicios y vendedores,"Subespacio ortogonal exacto de educación no reportada (MNAR = 100%): en los 3 modelos captura prácticamente la misma proporción de personas (~11.65%), validando que la bandera binaria evita distorsionar la escala ordinal sin contaminar los demás grupos."
Clúster 2,Profesionales Intensivos en Conocimiento y Alta Exigencia ESCO,27.99% · 42.75% (0.936x),9.47% · 34.34% (0.752x),38.85% · 47.96% (1.050x),2 · Profesionales científicos e intelectuales / 2 · Profesionales científicos e intelectuales,"Profesionales de alta complejidad cognitiva (ISCO 2, alto nivel universitario y mayor conteo de conocimientos ESCO). Presenta la MENOR reincidencia post-hoc en todos los modelos porque su movilidad es vertical (ascensos hacia códigos ESCO distintos). Ward aísla un núcleo más estricto y puro de alto conocimiento, mientras que K-Means asigna una frontera esférica más amplia."
Clúster 3,Trayectorias Consolidadas de Alta Rotación o Larga Permanencia,16.44% · 45.55% (0.997x),26.75% · 54.17% (1.186x),15.32% · 43.99% (0.963x),3 · Técnicos y profesionales de nivel medio / 3 · Técnicos y profesionales de nivel medio,"Trayectorias extensas (span_q alto), mayor duración acumulada y alto solapamiento contractual (overlap_ratio > 1.5). En Ward captura además la alta rotación con la tasa post-hoc de repetición más alta, mientras que K-Means separa estrictamente por amplitud temporal y permanencia prolongada."


---

## 6. Generación Append-Only de `Bitacora_Clustering_Jerarquico.md` y `Comparacion_A_vs_B_Clustering_Jerarquico.md` y Celda Guardia de Honestidad

En esta celda final:
1. Exportamos en modo **append-only** (conservando cualquier corrida previa como `[SUPERSEDED]` y marcando la corrida actual como `[CURRENT]`) los dos reportes Markdown:
   - `notebooks/refactor/Bitacora_Clustering_Jerarquico.md`
   - `notebooks/refactor/Comparacion_A_vs_B_Clustering_Jerarquico.md`
2. Ejecutamos las **4 verificaciones automáticas de la Celda Guardia**:
   - Integridad `SHA-256` de los 3 archivos de entrada (`CLEAN_VERSION_B`, `CLEAN_VERSION_A`, `SPLIT_MAP_PATH`).
   - Ausencia total de columnas de fuga (`reincide`, `veces_ese_oficio*`, `n_returns*`) en `V_PERSON` y `V_EXPERIENCE`.
   - Verificación matemática de estandarización (`StandardScaler`: media $= 0$, desviación estándar $= 1$ antes de calcular distancias euclidianas).
   - Verificador estricto de números en narrativa (`verify_narrative_numbers_against_tables`) sobre ambos archivos `.md`.

In [7]:
hash_verification_df = verify_input_hashes(CONFIG, INITIAL_HASHES)

# Verify exact standardization on the discovery matrix
assert np.allclose(person_bundle.X_discovery_scaled.mean(axis=0), 0.0, atol=1e-5), "StandardScaler mean != 0!"
assert np.allclose(person_bundle.X_discovery_scaled.std(axis=0), 1.0, atol=1e-5), "StandardScaler std != 1!"

bitacora_path = append_bitacora_jerarquico_md(
    config=CONFIG,
    run_id=CURRENT_RUN_ID,
    run_timestamp=CURRENT_RUN_TIMESTAMP,
    hash_df=hash_verification_df,
    leakage_df=leakage_audit_df,
    data_usage_df=data_usage_df,
    person_bundle=person_bundle,
    exp_bundle=exp_bundle,
    per_seed_df=per_seed_df,
    ab_summary_df=ab_summary_df,
)

comparacion_path = append_comparacion_ab_jerarquico_md(
    config=CONFIG,
    run_id=CURRENT_RUN_ID,
    run_timestamp=CURRENT_RUN_TIMESTAMP,
    data_usage_df=data_usage_df,
    per_seed_df=per_seed_df,
    ab_summary_df=ab_summary_df,
    person_bundle=person_bundle,
)

comparative_8_0_vs_8_1_path = write_comparative_analysis_8_0_vs_8_1_md(
    config=CONFIG,
    run_id=CURRENT_RUN_ID,
    run_timestamp=CURRENT_RUN_TIMESTAMP,
    expansion=expansion_result,
    person_bundle_3k=person_bundle,
)

# Negative test in RAM: tamper a number specifically in a narrative line and verify AssertionError is raised
tampered_text = bitacora_path.read_text(encoding="utf-8").replace(
    f"ARI = {person_bundle.ari_vs_kmeans:.4f}", "ARI = 0.9876"
)
try:
    verify_narrative_numbers_against_tables(tampered_text, doc_name="Tampered_RAM_Test")
    raise RuntimeError("Verifier did not fail on tampered number!")
except AssertionError as exc:
    negative_test_msg = f"CAPTURADO EXITOSAMENTE: {str(exc)[:140]}..."

# Positive tests on disk for both append-only .md files
ver_bitacora = verify_narrative_numbers_against_tables(
    bitacora_path.read_text(encoding="utf-8"),
    doc_name=bitacora_path.name,
)
ver_comparacion = verify_narrative_numbers_against_tables(
    comparacion_path.read_text(encoding="utf-8"),
    doc_name=comparacion_path.name,
)

guard_summary_df = pd.DataFrame(
    [
        {
            "archivo_md": "Prueba Negativa en RAM (Número Alterado Deliberadamente)",
            "numeros_narrativa_verificados": 1,
            "discrepancias_detectadas": 1,
            "estado_verificacion": negative_test_msg,
        },
        ver_bitacora,
        ver_comparacion,
    ]
)

display(Markdown("### 6.1 Verificación de Integridad SHA-256 de Insumos"))
display(hash_verification_df)

display(Markdown("### 6.2 Verificación Automática de Números en Narrativa (`Bitacora_Clustering_Jerarquico.md` y `Comparacion_A_vs_B_Clustering_Jerarquico.md`)"))
display(guard_summary_df)
print(
    f"Total figuras PNG exportadas: {len(figure_paths) + 1} | "
    f"Total tablas CSV exportadas: {len(exported_tables) + len(expansion_tables)} | "
    f"Informe comparativo generado: {comparative_8_0_vs_8_1_path.name}"
)

Total figuras PNG exportadas: 8 | Total tablas CSV exportadas: 13 | Informe comparativo generado: Analisis_Comparativo_8_0_vs_8_1_Clustering.md


### 6.1 Verificación de Integridad SHA-256 de Insumos

archivo_insumo,ruta_relativa,sha256_inicial,sha256_final,intacto_en_disco
clean_version_b,data/03_processed/empleos_limpio_con_competencias_sin_outliers.csv,c8ea36b51dea8803...,c8ea36b51dea8803...,Sí
clean_version_a,data/03_processed/empleos_unificado_limpio.csv,68cd34c624e8d23e...,68cd34c624e8d23e...,Sí
split_map,reports/eda_jobpath/tables/split_map_resume_id.csv,67dd3e3d68e1ff5e...,67dd3e3d68e1ff5e...,Sí


### 6.2 Verificación Automática de Números en Narrativa (`Bitacora_Clustering_Jerarquico.md` y `Comparacion_A_vs_B_Clustering_Jerarquico.md`)

archivo_md,numeros_narrativa_verificados,discrepancias_detectadas,estado_verificacion
Prueba Negativa en RAM (Número Alterado Deliberadamente),1,1,"CAPTURADO EXITOSAMENTE: [Tampered_RAM_Test] Narrative number verification FAILED! Numbers not found in tables of the same run: [""0.9876 (en: '- **Justificación del ..."
Bitacora_Clustering_Jerarquico.md,17,0,APROBADO (100% trazable a tablas de la misma corrida)
Comparacion_A_vs_B_Clustering_Jerarquico.md,16,0,APROBADO (100% trazable a tablas de la misma corrida)
